Calcualamos las variables con retrazo (lags de 7, 15 y 30 días) para que los clasificadores tengan contexto del pasado.

In [ ]:
import sqlite3
import pandas as pd

def calculo_variables_con_retraso(db_path: str) -> pd.DataFrame:
    """Conecta a SQLite, calcula las variables con retraso temporal (LAG)

    por boya geográfica y devuelve el DataFrame listo para el modelo.
    """
    try:
        # 1. Establecer la conexión con la base de datos SQLite
        conexion = sqlite3.connect(db_path)
        
        # 2. Definir la consulta SQL con la partición por coordenadas y orden cronológico
        # Usamos complete_date para asegurar el orden temporal correcto
        query = """
        SELECT 
            complete_date,
            latitude,
            longitude,
            zonal_winds,
            meridional_winds,
            sst AS sst_hoy,
            alert_anomaly,
            -- Trae el valor de SST de hace 7 registros para esa misma boya
            LAG(sst, 7) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_hace_7_dias,
            -- Trae el valor de SST de hace 15 registros para el horizonte predictor
            LAG(sst, 15) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_hace_15_dias
        FROM proyecto_final_elnino;
        """
        
        # 3. Ejecutar la consulta y cargarla directamente en un DataFrame
        df_resultado = pd.read_sql_query(query, conexion)
        
        # 4. Limpiar los registros iniciales que quedan con NaNs debido al desfase del LAG
        df_limpio = df_resultado.dropna().copy()
        
        print(f"✓ Variables calculadas con éxito. Registros útiles: {len(df_limpio):,}")
        return df_limpio

    except sqlite3.Error as e:
        print(f"Error al conectar o ejecutar la consulta en SQLite: {e}")
        return pd.DataFrame()
        
    finally:
        # Asegurar el cierre de la conexión si se llegó a abrir
        if 'conexion' in locals():
            conexion.close()

# --- EJECUCIÓN DEL BLOQUE ---
# Cambia 'proyecto_final_elnino' por el nombre exacto de tu archivo de base de datos
df_modelado = calculo_variables_con_retraso('proyecto_final_elnino')

if not df_modelado.empty:
    print(df_modelado.head())

       


DatabaseError: Execution failed on sql '
        SELECT 
            complete_date,
            latitude,
            longitude,
            zonal_winds,
            meridional_winds,
            sst AS sst_hoy,
            alert_anomaly,
            -- Trae el valor de SST de hace 7 registros para esa misma boya
            LAG(sst, 7) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_hace_7_dias,
            -- Trae el valor de SST de hace 15 registros para el horizonte predictor
            LAG(sst, 15) OVER(
                PARTITION BY latitude, longitude 
                ORDER BY complete_date
            ) AS sst_hace_15_dias
        FROM proyecto_final_elnino;
        ': no such table: proyecto_final_elnino